# Tahap 03: Data Preprocessing

Tahap ini merupakan inti dari prapemrosesan teks. Proses yang dilakukan mencakup pemuatan kamus eksternal (KBBI, *Slang*, *Stopword*, Ekstraksi Emoji), pembersihan teks lanjutan, normalisasi kata, *stemming* menggunakan Sastrawi, dan penyaringan token berdasarkan leksikon bahasa baku.

In [ ]:
# --- LIBRARIES & SETUP ---
import os
import re
import pandas as pd
import demoji
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Konfigurasi Direktori Utama (Sesuaikan dengan path lokal)
BASE_DIR = r'C:\Users\Asus\Desktop\SKRIPSI\1. MAIN'

# Inisialisasi Sastrawi Stemmer dan Stopword
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

sastrawi_sw_factory = StopWordRemoverFactory()
SASTRAWI_STOPWORD_SET = set(sastrawi_sw_factory.get_stop_words())

# Definisi Token Khusus
BADWORD_TOKEN = '[BADWORD]' 

print(f"Direktori Kerja: {BASE_DIR}")
print("Library Sastrawi dan Demoji siap digunakan.")

## 3.1 Pemuatan Dataset dan Kamus Leksikon Eksternal
Memuat data yang akan diproses beserta seluruh kamus referensi yang dibutuhkan untuk normalisasi dan *filtering*.

In [ ]:
# --- FUNGSI BANTUAN ---
def load_dict_from_csv(filename: str, key_col: str, value_col: str = None):
    """Memuat file CSV kamus dan mengkonversinya menjadi Set atau Dictionary."""
    file_path = os.path.join(BASE_DIR, filename)
    try:
        df = pd.read_csv(file_path)
        # Case Folding untuk Konsistensi Kamus
        df = df.apply(lambda x: x.astype(str).str.lower())
        
        if value_col:
            return dict(zip(df[key_col], df[value_col]))
        else:
            return set(df[key_col].unique())
    except FileNotFoundError:
        print(f"ERROR: File tidak ditemukan di {file_path}")
        return set() if not value_col else {}

# --- 1. PEMUATAN DATASET ---
DATA_FILE = 'data_labeling.csv'
DATASET_PATH = os.path.join(BASE_DIR, DATA_FILE)
try:
    df = pd.read_csv(DATASET_PATH)
except FileNotFoundError:
    print(f"FATAL ERROR: Dataset utama {DATA_FILE} tidak ditemukan!")
    df = pd.DataFrame()
    
# --- 2. PEMUATAN KAMUS LEKSIKON ---
PROFANITY_SET = load_dict_from_csv('profanity_dict.csv', 'profanity_list')
SLANG_DICT = load_dict_from_csv('slang_dict.csv', 'slang_list', 'baku')
KBBI_SET = load_dict_from_csv('kbbi_dict.csv', 'kbbi_list')
CUSTOM_STOPWORD_SET = load_dict_from_csv('stopword_dict.csv', 'stopword_list')

# Pemuatan Kamus Emoji Khusus
EMOJI_DF = pd.read_csv(os.path.join(BASE_DIR, 'emoji_dict.csv'))
EMOJI_MAP = dict(zip(EMOJI_DF['emoji_list'], EMOJI_DF['makna'].str.lower()))

# Penggabungan Stopword Sastrawi dan Kustom
FINAL_STOPWORD_SET = SASTRAWI_STOPWORD_SET.union(CUSTOM_STOPWORD_SET)

# Definisi kata kunci sentimen untuk penanganan kontradiksi emoji
NEGATIVE_TOKENS = {'sedih', 'marah', 'kecewa', 'disappointed', 'crying', 'angry', 'kesedihan'}
POSITIVE_TOKENS = {'bagus', 'cinta', 'bahagia', 'senang', 'tertawa', 'good'}

print(f"Dataset dimuat. Total baris awal: {len(df)}")
print(f"Ukuran Kamus KBBI (Set): {len(KBBI_SET)}")
print(f"Ukuran Stopword Final (Set): {len(FINAL_STOPWORD_SET)}")

## 3.2 Fungsi Normalisasi Teks Utama
Fungsi ini menangani penghapusan kata kasar, konversi emoji berbasis sentimen untuk menghindari kontradiksi label, *case folding*, dan koreksi kata tidak baku (*slang*).

In [ ]:
def clean_and_normalize(text: str, label: str) -> list:
    """
    Melakukan normalisasi teks meliputi penanganan kata kasar, konversi emoji,
    penghapusan karakter tidak relevan, dan koreksi bahasa gaul (slang).
    """
    processed_text = str(text)
    
    # 1. Netralisasi Kata Kasar (Profanity)
    for pword in PROFANITY_SET:
        processed_text = re.sub(r'\b' + re.escape(pword) + r'\b', BADWORD_TOKEN, processed_text, flags=re.I)
    
    # 2. Penghapusan Tautan, Username, dan Simbol Hastag
    processed_text = re.sub(r'https?://\S+|www\.\S+', '', processed_text)
    processed_text = re.sub(r'@[A-Za-z0-9]+', '', processed_text)
    processed_text = re.sub(r'#', '', processed_text)

    # 3. Konversi Emoji dengan Penanganan Kontradiksi Sentimen
    for emoji_char, token in EMOJI_MAP.items():
        is_contradictory = False
        
        is_token_negative = any(neg_word in token for neg_word in NEGATIVE_TOKENS)
        is_token_positive = any(pos_word in token for pos_word in POSITIVE_TOKENS)

        # Validasi kesesuaian makna emoji dengan label kelas
        if (label in ['Pujian', 'Saran']) and is_token_negative:
             is_contradictory = True
        elif (label == 'Keluhan') and is_token_positive:
             is_contradictory = True
             
        if is_contradictory:
             processed_text = processed_text.replace(emoji_char, '') 
        else:
             processed_text = processed_text.replace(emoji_char, f" {token} ") 

    # 4. Penghapusan Tanda Baca & Simbol
    processed_text = re.sub(r'[^a-zA-Z0-9\s\[\]]', ' ', processed_text)
    
    # 5. Case Folding
    processed_text = processed_text.lower()
    
    # 6. Tokenisasi
    tokens = processed_text.split()

    # 7 & 8. Standardisasi Karakter Berulang & Koreksi Slang
    normalized_tokens = []
    for token in tokens:
        if token == BADWORD_TOKEN.lower():
            normalized_tokens.append(BADWORD_TOKEN)
            continue
            
        # Menghapus huruf yang berulang lebih dari 2 kali (contoh: "baguuuus" -> "baguus")
        token = re.sub(r'(.)\1{2,}', r'\1\1', token) 
        
        # Konversi kata tidak baku menjadi baku
        token = SLANG_DICT.get(token, token) 
        normalized_tokens.append(token)
    
    return normalized_tokens

## 3.3 Eksekusi Normalisasi dan Penyaringan Awal

In [ ]:
# 1. Penghapusan Duplikasi Mutlak
initial_len = len(df)
df.drop_duplicates(subset=['ownerUsername', 'text'], keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Duplikasi dihapus: {initial_len - len(df)} baris.")

# 2. Penyaringan Indikasi Spam/Scam/Promosi
keywords_scam = ['scam', 'pinjol', 'promosi'] 
scam_mask = df['text'].str.contains('|'.join(keywords_scam), case=False, na=False)

df_final = df[~scam_mask].copy() 
print(f"Baris Scam/Promosi dihapus: {len(df) - len(df_final)} baris.")
print(f"Baris tersisa untuk diproses: {len(df_final)}")

# 3. Menerapkan Fungsi Pembersihan & Normalisasi Utama
df_final['tokens_normalized'] = df_final.apply(
    lambda row: clean_and_normalize(row['text'], row['label_pks']),
    axis=1 
)

## 3.4 Reduksi Dimensi Teks (Stopword, Stemming, & KBBI Filter)

In [ ]:
def reduce_dimension_and_filter(tokens: list) -> str:
    """Melakukan Stopword Removal, Stemming, dan validasi leksikon KBBI."""
    
    # 1. Stopword Removal
    tokens_no_stopword = [
        token for token in tokens if token not in FINAL_STOPWORD_SET
    ]
    
    # 2. Stemming (Sastrawi)
    stemmed_tokens = []
    for word in tokens_no_stopword:
        if word == BADWORD_TOKEN or any(t in word for t in EMOJI_MAP.values()):
            stemmed_tokens.append(word)
        else:
            stemmed_tokens.append(stemmer.stem(word)) 

    # 3. Filtering KBBI Lintas Bahasa
    tokens_filtered_kbbi = []
    for token in stemmed_tokens:
        if token == BADWORD_TOKEN or any(t in token for t in EMOJI_MAP.values()):
            tokens_filtered_kbbi.append(token)
            continue
        
        if token in KBBI_SET:
            tokens_filtered_kbbi.append(token)

    # 4. Filter Token Kosong & Karakter Tunggal
    final_tokens = [
        token for token in tokens_filtered_kbbi if len(token) > 1 and token.strip() != ''
    ]
    
    return ' '.join(final_tokens)

# Menerapkan pipeline reduksi
df_final['text_processed'] = df_final['tokens_normalized'].apply(reduce_dimension_and_filter)

# Menghapus kolom token sementara
df_final.drop(columns=['tokens_normalized'], inplace=True)

# Membuang baris yang menjadi kosong akibat filtering
initial_len_final = len(df_final)
df_final = df_final[df_final['text_processed'].str.strip() != '']
baris_kosong_dihapus = initial_len_final - len(df_final)

print(f"Baris yang menjadi kosong setelah filtering: {baris_kosong_dihapus} baris.")
print(f"Total baris data bersih (Final): {len(df_final)}")

## 3.5 Penyimpanan Data Bersih
Menyimpan hasil akhir prapemrosesan teks ke dalam format CSV untuk digunakan pada tahap ekstraksi fitur dan pemodelan.

In [ ]:
print("\nPREPROCESSING SELESAI.")
print("Contoh Data Hasil Akhir (5 Baris Teratas):")

# Pratinjau data sebelum penyimpanan
df_temp_display = df_final.copy()
df_temp_display['text_processed_display'] = df_final['text_processed']

print(df_temp_display[['text', 'label_pks', 'text_processed_display']].head(5))
print("-" * 50)

# Menimpa kolom teks asli dengan teks yang sudah dibersihkan
df_final['text'] = df_final['text_processed']
df_final.drop(columns=['text_processed'], inplace=True)

# --- PENYIMPANAN KE FILE CSV ---
OUTPUT_FILE = 'data_preprocessing_2.0.csv'
OUTPUT_PATH = os.path.join(BASE_DIR, OUTPUT_FILE)

# Mendefinisikan kolom yang akan dipertahankan
COLUMNS_TO_SAVE = [
    'id', 'timestamp', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 
    'ownerUsername', 'text', 'label_pks' 
]

existing_columns = [col for col in COLUMNS_TO_SAVE if col in df_final.columns]

df_final[existing_columns].to_csv(OUTPUT_PATH, index=False)
print(f"Data final ({len(df_final)} baris) berhasil disimpan ke: {OUTPUT_PATH}")